# 🚀 Credit Card Dispute Resolver — Backend on Colab

**Before running anything:** Go to `Runtime → Change runtime type → T4 GPU`

Then run cells in order: **1 → 2 → 3 → 4**.

In [1]:
# ── CELL 1 — Install dependencies ─────────────────────────────────────────────

!pip install -q --upgrade pip

!pip install -q \
fastapi \
"uvicorn[standard]" \
pydantic \
python-multipart \
python-dotenv \
torch \
transformers \
sentence-transformers \
scikit-learn==1.6.1 \
joblib \
numpy \
chromadb \
langchain \
langchain-core \
langchain-community \
langchain-huggingface \
langchain-chroma \
langchain-groq \
pyngrok

import torch
import numpy as np
import transformers
import sentence_transformers
import chromadb

print("✅ Dependencies installed")
print(f"numpy                  : {np.__version__}")
print(f"torch                  : {torch.__version__}")
print(f"transformers           : {transformers.__version__}")
print(f"sentence-transformers  : {sentence_transformers.__version__}")
print(f"chromadb               : {chromadb.__version__}")
print(f"GPU available          : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name               : {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 41.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.4.0 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.4.0 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.
✅ Dependencies installed
numpy                  : 2.0.2
torch                  : 2.11.0+cu128
transformers           : 5.13.1
sentence-transformers  : 5.6.0
chromadb               : 1.5.9
GPU available          : True
GPU name               : Tesla T4


In [2]:
# Mount Google Drive and configure backend path
from google.colab import drive
import os, sys

drive.mount('/content/drive')

BACKEND_DIR='/content/drive/MyDrive/backend'
MODEL_BASE=os.path.join(BACKEND_DIR,'models')

os.environ['MODEL_BASE_PATH']=MODEL_BASE

if BACKEND_DIR not in sys.path:
    sys.path.insert(0,BACKEND_DIR)

print('Backend:',BACKEND_DIR)
print('Models :',MODEL_BASE)
assert os.path.exists(BACKEND_DIR), 'Backend folder not found'
assert os.path.exists(MODEL_BASE), 'Models folder not found'


Mounted at /content/drive
Backend: /content/drive/MyDrive/backend
Models : /content/drive/MyDrive/backend/models


In [3]:
# Import backend directly from Google Drive
from main import app

print("Backend imported successfully.")


Backend imported successfully.


In [ ]:
# ── CELL 4 — Launch FastAPI + ngrok ───────────────────────────────────────────
import os
import traceback
from google.colab import userdata

NGROK_TOKEN = userdata.get('NGROK_TOKEN')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

import time, requests, asyncio, threading
import uvicorn
from pyngrok import ngrok, conf

def run_server():
    import asyncio
    import traceback
    import uvicorn

    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

        config = uvicorn.Config(
            "main:app",
            host="0.0.0.0",
            port=8000,
            log_level="info",
            loop="asyncio",
        )

        server = uvicorn.Server(config)
        loop.run_until_complete(server.serve())

    except Exception:
        print("=" * 60)
        print("UVICORN STARTUP FAILED")
        print("=" * 60)
        traceback.print_exc()

conf.get_default().auth_token = NGROK_TOKEN
ngrok.kill()
time.sleep(1)

# Start server thread before ngrok so port 8000 is ready
t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(5)   # give uvicorn time to bind

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url

with open('/content/public_url.txt', 'w') as f:
    f.write(PUBLIC_URL)

print('='*60)
print('  🚀 BACKEND IS LIVE!')
print('='*60)
print(f'  Public URL : {PUBLIC_URL}')
print(f'  API Docs   : {PUBLIC_URL}/docs')
print(f'  Health     : {PUBLIC_URL}/health')
print('='*60)
print()
print('  📋 Paste into frontend/vite.config.js:')
print(f'     target: "{PUBLIC_URL}"')
print()
print('  ⚠️  Do NOT stop this cell — stopping kills the server')
print()

# Self-test
time.sleep(3)
try:
    r = requests.get(f"{PUBLIC_URL}/health", timeout=100)

    print("HTTP status:", r.status_code)
    print("Response:")
    print(r.text)

    if r.ok:
        h = r.json()
        print(h)
    # h = requests.get(f'{PUBLIC_URL}/health', timeout=20).json()
    print('  Self-test results:')
    print(f'    status   : {h["status"]}')
    print(f'    bert     : {h["models_loaded"]["bert"]}')
    print(f'    baseline : {h["models_loaded"]["baseline"]}')
    print(f'    rag_ready: {h["rag_ready"]}')
    print(f'    cuda     : {h["cuda"]}')
    print()
    print('  ✅ Server is healthy!')
except Exception as e:
    print(f'  ⚠️  Health check failed: {e}')
    print('  Wait 15 seconds then try: requests.get(PUBLIC_URL + "/health").json()')
    traceback.print_exc()

# Keep cell alive
while True:
    time.sleep(300)
    print(f'  ⏱️  Still running — {PUBLIC_URL}')


INFO:     Started server process [560]
INFO:     Waiting for application startup.


Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

  🚀 BACKEND IS LIVE!
  Public URL : https://afterlife-crudeness-spent.ngrok-free.dev
  API Docs   : https://afterlife-crudeness-spent.ngrok-free.dev/docs
  Health     : https://afterlife-crudeness-spent.ngrok-free.dev/health

  📋 Paste into frontend/vite.config.js:
     target: "https://afterlife-crudeness-spent.ngrok-free.dev"

  ⚠️  Do NOT stop this cell — stopping kills the server



/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.7.2 when using version 1.6.1. This might lead to breaking c

HTTP status: 502
Response:
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymous" 

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
  ⏱️  Still running — https://afterlife-crudeness-spent.ngrok-free.dev
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
  ⏱️  Still running — https://afterlife-crudeness-spent.ngrok-free.dev
INFO:     61.1.167.86:0 - "OPTIONS /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "GET /health HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "OPTIONS /predict-full HTTP/1.1" 200 OK
INFO:     61.1.167.86:0 - "POST /predict-ful